# Data Cleaning: Hacker News "Who is hiring?" Posts
**CSC 366: Principles of Data Science | Group 02**

This notebook cleans the Hacker News (HN) job posts collected from 24 monthly "Who is hiring?" threads (Nov 2024 to Oct 2026). The output, `hn_clean.csv`, uses the same column names as `serp_clean.csv` so the two can be merged in the next notebook. Skills and years of experience are extracted after merging.



**Upload `raw_hn_24m.json` to the Colab files panel before running.**

## Load the data

In [1]:
import pandas as pd
import numpy as np
import json
import re      # regular expressions (pattern matching in text)
import html    # converts HTML codes like &#x2F; back into normal characters

with open("raw_hn_24m.json") as f:
    hnPosts = json.load(f)

dfHN = pd.DataFrame(hnPosts)
dfHN.head()

,post_id,author,created_at,text,thread_title
0,49922584,mwest1066,2026-10-01T15:02:37.000Z,PrairieLearn (Remote US) — Full-Stack Software...,Ask HN: Who is hiring? (October 2026)
1,49922593,shepherd-eng,2026-10-01T15:02:51.000Z,"Shepherd (Series B) | ONSITE | San Francisco, ...",Ask HN: Who is hiring? (October 2026)
2,49922609,mcintyem,2026-10-01T15:03:36.000Z,"Sidekick Labs | Senior SWE, R&amp;D Engineer, ...",Ask HN: Who is hiring? (October 2026)
3,49922615,eterps,2026-10-01T15:04:09.000Z,Open Education Applications &#x2F; Neon | Seni...,Ask HN: Who is hiring? (October 2026)
4,49922616,pagnos,2026-10-01T15:04:11.000Z,"PAGNOS | Cloud Engineer, AWS Serverless (TypeS...",Ask HN: Who is hiring? (October 2026)


## First look

In [6]:
print(f"The number of column: {dfHN.shape[1]}")
print(f"The number of rows: {dfHN.shape[0]}")
dfHN.info()
dfHN.iloc[0].text

The number of column: 5
The number of rows: 7627
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7627 entries, 0 to 7626
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   post_id       7627 non-null   int64 
 1   author        7626 non-null   object
 2   created_at    7627 non-null   object
 3   text          7626 non-null   object
 4   thread_title  7627 non-null   object
dtypes: int64(1), object(4)
memory usage: 298.1+ KB


'PrairieLearn (Remote US) — Full-Stack Software Engineer — TypeScript &#x2F; Postgres &#x2F; React &#x2F; AI<p>PrairieLearn (<a href="https:&#x2F;&#x2F;www.prairielearn.com" rel="nofollow">https:&#x2F;&#x2F;www.prairielearn.com</a>) is an open-source assessment platform used by universities across the US (Berkeley, Princeton, Michigan, Illinois, and others). We power mastery-based learning and large-scale exams.<p>We’re a small, profitable, early-stage company (bootstrapped, no VC) that is growing quickly. Our users love us and we have very high retention and rapid spread through word of mouth. As an early-stage hire, you’ll work across the stack and enjoy meaningful ownership from day one.<p>Tech we use: Node.js &#x2F; TypeScript backend, Postgres, AWS, React. We’re also developing AI tooling, including LLM agents to help instructors create content, and vision-language models to help grade student work. PrairieLearn is open core: <a href="https:&#x2F;&#x2F;github.com&#x2F;PrairieLearn

7,627 posts and only 5 columns. All of the job information (company, role, location, salary) is inside `text`.

## Remove the empty post


In [11]:
# Remove the post with no text (nothing to clean or analyze)
print(f"Rows before: {len(dfHN)}")
dfHN = dfHN[dfHN["text"].notna()].copy()
print(f"Rows after: {len(dfHN)}")

Rows before: 7627
Rows after: 7626


## Clean the HTML

clean HTML tags

In [12]:
dfHN["text"].iloc[0]

'PrairieLearn (Remote US) — Full-Stack Software Engineer — TypeScript &#x2F; Postgres &#x2F; React &#x2F; AI<p>PrairieLearn (<a href="https:&#x2F;&#x2F;www.prairielearn.com" rel="nofollow">https:&#x2F;&#x2F;www.prairielearn.com</a>) is an open-source assessment platform used by universities across the US (Berkeley, Princeton, Michigan, Illinois, and others). We power mastery-based learning and large-scale exams.<p>We’re a small, profitable, early-stage company (bootstrapped, no VC) that is growing quickly. Our users love us and we have very high retention and rapid spread through word of mouth. As an early-stage hire, you’ll work across the stack and enjoy meaningful ownership from day one.<p>Tech we use: Node.js &#x2F; TypeScript backend, Postgres, AWS, React. We’re also developing AI tooling, including LLM agents to help instructors create content, and vision-language models to help grade student work. PrairieLearn is open core: <a href="https:&#x2F;&#x2F;github.com&#x2F;PrairieLearn

**AI assistance:** The raw Hacker News data was shared with AI (Claude) before cleaning. Claude counted every HTML tag and code in the posts (only `<p>`, `<a>`, `<i>`, `<pre>`, `<code>`, and six codes such as `&#x2F;` and `&amp;` appear), so we knew what to clean directly.

In [13]:
def cleanText(rawText):
    text = rawText.replace("<p>", "\n")        # paragraph tags -> line breaks
    text = re.sub(r"<[^>]+>", "", text)        # remove every other tag, like <a href=...>
    text = html.unescape(text)                 # &#x2F; -> /   &amp; -> &   &#x27; -> '
    return text.strip()

In [17]:
dfHN["description"] = dfHN["text"].apply(cleanText)
dfHN["description"].iloc[0][:100]

'PrairieLearn (Remote US) — Full-Stack Software Engineer — TypeScript / Postgres / React / AI\nPrairie'

## Flag comments that are not job posts

Hacker News is a public discussion forum, similar to Reddit, where users can post and comment freely. Because of this, not every top-level comment is an actual job listing. Some are complaints, jokes, or moderator removals.

To help filter these out, posts under 100 characters were flagged with `is_job_post = False`. We kept these posts in the dataset just in case we needed them later.


In [15]:
# Posts under 100 characters are flagged as not job posts
dfHN["is_job_post"] = dfHN["description"].str.len() >= 100

print(f"Flagged as not a job post: {(dfHN['is_job_post'] == False).sum()}")

# Example of what was caught
dfHN[dfHN["is_job_post"] == False]["description"].head(5)

Flagged as not a job post: 80


,description
38,there used to be 800+ posts here :(
57,This role always shows up here. There's no way...
152,Koop AI | Senior Software Engineer | NYC/JC Hy...
205,"telli hiring in Eng, GTM, Design [Berlin] - ht..."
239,https://grnh.se/rlbk86k09us\nHiring for multip...


## 5. Header and company

Since the raw data was shared with AI (Claude) ahead of time, Claude checked how the posts were formatted and found that about 91% start with a header line like `Company | Role | Location | Full-time | Salary`. This shaped how the header is split. The company is almost always the first piece, so it is split out. A few headers start with a `|`, so leading `|` characters are stripped first.

The other pieces are not split out by position, because Claude's check showed their order is not consistent. For example, `Shepherd | ONSITE | San Francisco` has the location third, while `OpenRent | London, UK | Full-Time` has it second. Instead, the remaining columns (role, location, job type) are found by searching the header for keywords.

In [18]:
# The first line of each post is the header: "Company | Role | Location | ..."
dfHN["header"] = dfHN["description"].str.split("\n").str[0]

# Create a "company_name" column from the header:
#   1. strip("| ")  -> remove "|" and spaces from both ends (some headers start with "|")
#   2. split("|")   -> cut the header into pieces at each "|"
#   3. str[0]       -> keep the first piece (the company)
#   4. strip()      -> remove leftover spaces
dfHN["company_name"] = dfHN["header"].str.strip("| ").str.split("|").str[0].str.strip()

dfHN[["company_name", "header"]].head(10)

,company_name,header
0,PrairieLearn (Remote US) — Full-Stack Software...,PrairieLearn (Remote US) — Full-Stack Software...
1,Shepherd (Series B),"Shepherd (Series B) | ONSITE | San Francisco, ..."
2,Sidekick Labs,"Sidekick Labs | Senior SWE, R&D Engineer, Inte..."
3,Open Education Applications / Neon,Open Education Applications / Neon | Senior/Le...
4,PAGNOS,"PAGNOS | Cloud Engineer, AWS Serverless (TypeS..."
5,Snout https://snout.com/,Snout https://snout.com/ | Multiple Engineerin...
6,Planlab.ai,Planlab.ai | | Product Engineer (Design) & Pla...
7,OpenRent,"OpenRent | London, UK | Full-Time | ONSITE+PAR..."
8,CareerJumpShip,CareerJumpShip | Multiple engineering + go-to-...
9,Checkly,Checkly | https://www.checklyhq.com | Senior P...


## Posted Dates

HN gives a real posting date, and the thread title gives the month.

**AI assistance:** The regular expression used to extract the month from the thread title (`\((.+)\)`) was written with help from AI (Claude).

In [19]:
# HN gives a real posting date (Can use datetime for better tracking)
dfHN["posted_date"] = pd.to_datetime(dfHN["created_at"]).dt.date

# The month comes from the thread title, e.g. "Ask HN: Who is hiring? (February 2026)"
dfHN["thread_month"] = dfHN["thread_title"].str.extract(r"\((.+)\)")[0]

dfHN["thread_month"].value_counts().head()

,count
thread_month,
February 2026,410
February 2025,396
November 2025,392
December 2025,377
June 2025,358


## Job types (intern, contract, etc...)

Both are found by checking the header for keywords. If no job type keyword appears, the job type is `"Unknown"`.

In [20]:
# Job type: check the header for each kind of keyword
def getJobTypeHN(header):
    header = header.lower()
    if "intern" in header:
        return "Internship"
    if "contract" in header:
        return "Contractor"
    if "part-time" in header or "part time" in header:
        return "Part-time"
    if "full-time" in header or "full time" in header or "fulltime" in header:
        return "Full-time"
    return "Unknown"

In [22]:
headerLower = dfHN["header"].str.lower()

# Finding if the job is remote
dfHN["is_remote"] = headerLower.str.contains("remote", regex=False)

dfHN["job_type"] = dfHN["header"].apply(getJobTypeHN)

print(dfHN["is_remote"].value_counts())
dfHN["job_type"].value_counts()

is_remote
True     3933
False    3693
Name: count, dtype: int64


,count
job_type,
Full-time,4577
Unknown,2649
Contractor,202
Internship,116
Part-time,82


## Senior flag

Posts are flagged if any whole word is in the senior word list. Whole words are used so that words like "leadership" or "SRE" are not flagged.

In [23]:
# AI assistance: Claude helped identify words that indicate senior-level jobs
# and punctuation to clean up, based on job titles in our dataset
# (e.g., "Sr.", "Senior/Staff", "Lead - Backend").
def isSenior(title):
    title = title.lower()
    seniorWords = ["senior", "sr", "staff", "lead", "principal"]

    # Turn punctuation into spaces so "Sr." -> "sr" and "Senior/Staff" -> "senior staff"
    for symbol in [".", ",", "/", "-", "(", ")", ":", "|", "&"]:
        title = title.replace(symbol, " ")

    for word in title.split():
        if word in seniorWords:
            return True
    return False

In [24]:
dfHN["is_senior"] = dfHN["header"].apply(isSenior)

print(f"Senior posts flagged: {dfHN['is_senior'].sum()}")
dfHN[dfHN["is_senior"]]["header"].head(10)

Senior posts flagged: 2114


,header
2,"Sidekick Labs | Senior SWE, R&D Engineer, Inte..."
3,Open Education Applications / Neon | Senior/Le...
9,Checkly | https://www.checklyhq.com | Senior P...
13,Mainmatter (https://mainmatter.com/rust-consul...
15,"Fastly | Software Engineers (Senior, Staff, Pr..."
16,"Arrangr | Growth Lead, Expansion & Lifecycle |..."
17,River | Staff Engineers (Elixir) | NYC or REMO...
18,"JustPark | Senior Software Engineer, Reporting..."
20,Fazeshift | Senior Software Engineer | San Fra...
24,Starbridge | Senior Product Engineer | NYC or ...


**Limitation** HN leans toward startups hiring experienced engineers.



## US vs. non-US

Hacker News includes jobs from around the world, but this project is about US jobs. Each post is labeled based on its header:

- **"US"**: the header mentions a US code (US, USA, NYC, SF, or a state code like ", CA") or a US city or state name.
- **"Non-US"**: the header mentions a place outside the US, or a € or £ sign.
- **"Unknown"**: the header mentions neither (for example "Remote (Global)").

US is checked first, so a post with offices in both the US and another country counts as US. Short codes like "US" are matched only in capital letters, so the word "us" in a sentence is not counted.

**AI assistance:** AI (Claude) was used to create the lists and regex patterns used for this step: the state codes (`stateCodes`), the US codes (`usCodes`), and the US and non-US place names (`usPlaces`, `nonUsPlaces`).

It also suggested checking for US locations first, so that posts with offices in both the US and another country still count as US jobs.

In [25]:
# Signals that a post is in the US. Short codes are checked case-sensitive
# (so "US" doesn't match the word "us"); place names are checked in lowercase.
stateCodes = r",\s?(AL|AK|AZ|AR|CA|CO|CT|DE|FL|GA|HI|ID|IL|IN|IA|KS|KY|LA|ME|MD|MA|MI|MN|MS|MO|MT|NE|NV|NH|NJ|NM|NY|NC|ND|OH|OK|OR|PA|RI|SC|SD|TN|TX|UT|VT|VA|WA|WV|WI|WY|DC)\b"
usCodes = r"\bUS\b|\bUSA\b|\bU\.S\.|\bSF\b|\bNYC\b|" + stateCodes
usPlaces = ["united states", "san francisco", "bay area", "new york", "seattle", "austin",
            "boston", "chicago", "denver", "los angeles", "palo alto", "mountain view",
            "san jose", "california", "texas", "virginia", "massachusetts", "colorado",
            "illinois", "new jersey", "north carolina", "georgia", "florida", "oregon",
            "utah", "pennsylvania", "washington"]

# Signals that a post is outside the US
nonUsPlaces = ["uk", "london", "england", "germany", "berlin", "munich", "europe", "eu",
               "emea", "canada", "toronto", "vancouver", "montreal", "india", "bangalore",
               "amsterdam", "netherlands", "paris", "france", "spain", "barcelona", "madrid",
               "switzerland", "zurich", "sweden", "stockholm", "denmark", "copenhagen",
               "ireland", "dublin", "poland", "australia", "sydney", "singapore", "japan",
               "tokyo", "israel", "tel aviv", "brazil", "mexico", "portugal", "lisbon",
               "finland", "helsinki", "norway", "oslo", "austria", "vienna", "italy", "estonia"]

def getRegion(header):
    lower = " " + header.lower().replace("|", " ").replace(",", " ").replace("(", " ").replace(")", " ").replace("/", " ") + " "

    # US first: a post listing both US and non-US locations still has US jobs
    if re.search(usCodes, header):
        return "US"
    for place in usPlaces:
        if " " + place + " " in lower:
            return "US"

    for place in nonUsPlaces:
        if " " + place + " " in lower:
            return "Non-US"
    if "€" in header or "£" in header:
        return "Non-US"

    return "Unknown"

dfHN["region"] = dfHN["header"].apply(getRegion)

dfHN["region"].value_counts()

,count
region,
US,4168
Unknown,1763
Non-US,1695


## State

Many HN posts name a city without a state ("San Francisco", "NYC"), we use a dictionary to store those.

- US posts with no state are labeled `"Remote"` or `"Unknown"`.
- Posts outside the US are labeled `"Not US"`
- Posts whose region is unknown are labeled `"Unknown"` (an unknown region is not the same as outside the US).

**Limitation:** when a post lists several cities, only the first match is used.

In [26]:
dfHN["state"] = dfHN["header"].str.extract(stateCodes)[0]

# Many HN posts name a city without a state code ("San Francisco", "NYC")
# AI assistance: AI (Claude) was used to create this city-to-state list,
# covering the major US tech cities that appear most often in the HN posts.
cityToState = {
    "san francisco": "CA", "bay area": "CA", "los angeles": "CA", "palo alto": "CA",
    "mountain view": "CA", "san jose": "CA", "new york": "NY", "nyc": "NY",
    "seattle": "WA", "austin": "TX", "boston": "MA", "chicago": "IL", "denver": "CO",
}

def getStateFromCity(header):
    header = header.lower()
    for city in cityToState:
        if city in header:
            return cityToState[city]
    return None

dfHN["state"] = dfHN["state"].fillna(dfHN["header"].apply(getStateFromCity))

# US posts with no state: remote if the header says so, otherwise unknown
usNoState = (dfHN["region"] == "US") & (dfHN["state"].isnull())
dfHN.loc[usNoState & dfHN["is_remote"], "state"] = "Remote"
dfHN.loc[usNoState & (dfHN["is_remote"] == False), "state"] = "Unknown"

# Posts outside the US don't get a state; posts with an unknown region get "Unknown"
dfHN.loc[dfHN["region"] == "Non-US", "state"] = "Not US"
dfHN.loc[dfHN["region"] == "Unknown", "state"] = "Unknown"

dfHN["state"].value_counts().head(15)

,count
state,
Unknown,2003
Not US,1695
Remote,1253
CA,1078
NY,959
TX,131
WA,110
MA,104
IL,58


## Salary

 Salaries can appear anywhere in a post and are written in many different ways: `$170k to $230k`, `$190–250k`, or `$120,000 - $150,000`.


**AI assistance:** AI (Claude) was used to write the regex pattern that finds these salary ranges.

In [27]:
# A salary range: a $ amount, a dash or "to", then another amount, with optional k
salaryPattern = r"(\$\s?\d[\d,.]*\s?[kK]?\s?(?:-|–|—|to)\s?\$?\s?\d[\d,.]*\s?[kK]?)"

dfHN["salary_text"] = dfHN["description"].str.extract(salaryPattern)[0]

# Alot of the post don't mention a salary range, so we used droped them
dfHN["salary_text"].dropna().head(15)

,salary_text
0,$100k-$180k
2,$170k to $230k
8,$104K–$150K
11,$215K-$325K
17,$200K-$250K
19,$130-240K
20,$200k–$235k
27,$180k-$250k
29,$110k–$159k
30,"$260,000 - $350,000"


## Salary part 2

- Split the range into a low and a high number.
- Remove `$` and commas.
- If there's a `k`, multiply by 1,000.

**Two extra cases:** In `$190–250k`, the `k` is only written on the high end, and in `$180k - $210`, it is only written on the low end. In both cases, the `k` is applied to both numbers, but only to a number under 1,000, so `$178.5k - $243,800` stays \$243,800.

**AI assistance:** AI (Claude) was used to identify and handle these special cases, including the rule that a `k` is only applied to numbers under 1,000. That rule was added after a review found a salary that had been read as \$243.8 million.

In [28]:
def toNumberHN(text):
    text = text.replace("$", "").replace(",", "").strip().lower()
    if text.endswith("k"):
        return float(text.replace("k", "")) * 1000
    return float(text)

def parseSalaryHN(salaryText):
    if pd.isna(salaryText):
        return None, None

    # Make every kind of dash (and "to") the same so one split works
    text = salaryText.replace("–", "-").replace("—", "-").replace(" to ", "-")
    pieces = text.split("-")
    low = pieces[0].strip()
    high = pieces[-1].strip()

    # "$190-250k": the k on the high end also applies to the low end
    # (only when the low number is small, so "$90,000-120k" is not multiplied twice)
    if high.lower().endswith("k") and not low.lower().endswith("k") and toNumberHN(low) < 1000:
        low = low + "k"

    # "$180k - $210": the k on the low end also applies to the high end
    # (only when the high number is small, so "$178.5k - $243,800" stays $243,800)
    if low.lower().endswith("k") and not high.lower().endswith("k") and toNumberHN(high) < 1000:
        high = high + "k"

    return toNumberHN(low), toNumberHN(high)


In [29]:
salaryResults = dfHN["salary_text"].apply(parseSalaryHN)
dfHN[["salary_min", "salary_max"]] = pd.DataFrame(salaryResults.tolist(), index=dfHN.index)

dfHN["salary_min"].describe()

,salary_min
count,1356.000000
mean,143223.851586
std,48323.885526
min,0.000000
25%,120000.000000
50%,150000.000000
75%,175000.000000
max,300000.000000


## Salary Part 3

Hacker News posts usually don't say whether a salary is yearly, monthly, or hourly.

We decided on the threshold of \$20,000.This was based on the US federal minimum wage (\$7.25 an hour).

Any salary at or above \$20,000 is treated as yearly.

Anything below that can't be a full-time yearly salary, so it must be hourly, monthly, or a typo. If the post doesn't mention it, they are labeled `"unclear"` and left out.

A range is also labeled `"unclear"` if the maximum is smaller than the minimum (a typo like `$160k - 20k`) or if it is above \$1,000,000.

**AI assistance:** AI (Claude) was used to help find the special cases in the salary data, such as typos, hourly or monthly amounts, and impossible values, and to decide how each one should be handled.

In [30]:
# Under $20,000 can't be a yearly salary: it's hourly ("$23-$34"), monthly ("$3,500-$5,000"),
# or a typo. HN doesn't say which, so these are marked "unclear" instead of guessed.
# Also unclear: a range where the max is smaller than the min (a typo like "$160k - 20k")
# Also unclear: a max over $1,000,000 (not a plausible salary)
looksYearly = (dfHN["salary_min"] >= 20000) & (dfHN["salary_max"] >= dfHN["salary_min"]) & (dfHN["salary_max"] <= 1000000)

dfHN["salary_unit"] = None
dfHN.loc[looksYearly, "salary_unit"] = "a year"
dfHN.loc[dfHN["salary_min"].notna() & (looksYearly == False), "salary_unit"] = "unclear"

# Yearly columns (same names as SerpAPI) only for clear yearly salaries
dfHN["salary_min_yearly"] = dfHN["salary_min"].where(looksYearly)
dfHN["salary_max_yearly"] = dfHN["salary_max"].where(looksYearly)

print(dfHN["salary_unit"].value_counts(dropna=False))
dfHN[dfHN["salary_unit"] == "unclear"]["salary_text"].head(10)

salary_unit
None       6270
a year     1309
unclear      47
Name: count, dtype: int64


,salary_text
138,$23-$34
149,$150 - $300
172,"$3,500-$5,000"
175,$50–75
251,$7.25–$10
285,$120-160
307,$90–$110
467,$23-$34
615,$80–140
657,$6–$253K


## Duplicates

Many companies post the same ad every month. A post with the exact same header as an earlier one is treated as the same posting. Posts are sorted by date first, so the **first** time an ad appeared is kept as the original and later reposts are flagged.

In [31]:
# Same company reposting the same header in different months = same posting
dupCheck = dfHN[["header"]].copy()
dupCheck["header"] = dupCheck["header"].str.lower().str.strip()

# Sort oldest first so the FIRST time a posting appeared is the one kept
dfHN = dfHN.sort_values("posted_date")
dupCheck = dupCheck.loc[dfHN.index]

dfHN["is_duplicate"] = dupCheck.duplicated(keep="first")

print(f"Duplicates flagged: {dfHN['is_duplicate'].sum()}")
print(f"Unique posts: {(dfHN['is_duplicate'] == False).sum()}")

Duplicates flagged: 1923
Unique posts: 5703


## Salary outlier flag

The fences are built from US, full-time, non-senior, non-duplicate job posts, then applied to every row.

In [ ]:
isUS = dfHN["region"] == "US"
isFullTime = dfHN["job_type"] == "Full-time"
notSenior = dfHN["is_senior"] == False
notDuplicate = dfHN["is_duplicate"] == False
isJobPost = dfHN["is_job_post"] == True

baseSalaries = dfHN[isUS & isFullTime & notSenior & notDuplicate & isJobPost]["salary_min_yearly"]

Q1 = baseSalaries.quantile(0.25)
Q3 = baseSalaries.quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Salaries used for fences: {baseSalaries.notna().sum()}")
print(f"Q1 = {Q1}, Q3 = {Q3}, IQR = {IQR}")
print(f"fences: {lower} and {upper}")

tooLow = dfHN["salary_min_yearly"] < lower
tooHigh = dfHN["salary_min_yearly"] > upper
dfHN["is_salary_outlier"] = tooLow | tooHigh

print(f"Salary outliers flagged: {dfHN['is_salary_outlier'].sum()}")
dfHN[dfHN["is_salary_outlier"]][["company_name", "salary_text", "region", "is_senior"]].head(15)

Salaries used for fences: 297
Q1 = 125000.0, Q3 = 175000.0, IQR = 50000.0
fences: 50000.0 and 250000.0
Salary outliers flagged: 35


,company_name,salary_text,region,is_senior
6775,Ontic,$275k - $300k,US,False
6735,Levels.fyi,$30-50k,Non-US,False
6969,Wyndly,$45K - $120K,Unknown,False
6497,Wyndly,$45K - $60K,Unknown,False
6610,Cash App / Afterpay (Block),"$276,800—$415,200",US,True
6567,Roblox,$289-338k,US,True
6213,Peanut,$40-110k,Unknown,False
5921,Proof-of-skill (proofofskill.org),$25-30k,Unknown,False
5449,Levels.fyi,$30-50k,Non-US,False
5350,Mechanize,$300k-$600k,US,False


The fences here are \$50,000 and \$250,000

## Save

The columns are renamed to match `serp_clean.csv` so the two files can be stacked in the merge notebook:
- `post_id` becomes `job_id`
- `header` becomes `title` (HN has no separate title field)
- `location` is left empty because HN has no separate location field (`state` and `region` hold the location information)

HN-only columns (`region`, `posted_date`, `thread_month`, `is_job_post`) are kept.

**AI assistance:** AI (Claude) was used to keep the column names consistent between the Hacker News and SerpAPI datasets, so that both files could be combined without renaming columns in the merge notebook.

In [ ]:
dfHNClean = pd.DataFrame({
    "source": "hackernews",
    "job_id": dfHN["post_id"],
    "title": dfHN["header"],
    "company_name": dfHN["company_name"],
    "location": None,
    "state": dfHN["state"],
    "region": dfHN["region"],
    "is_remote": dfHN["is_remote"],
    "job_type": dfHN["job_type"],
    "is_senior": dfHN["is_senior"],
    "salary_text": dfHN["salary_text"],
    "salary_unit": dfHN["salary_unit"],
    "salary_min": dfHN["salary_min"],
    "salary_max": dfHN["salary_max"],
    "salary_min_yearly": dfHN["salary_min_yearly"],
    "salary_max_yearly": dfHN["salary_max_yearly"],
    "is_salary_outlier": dfHN["is_salary_outlier"],
    "posted_date": dfHN["posted_date"],
    "thread_month": dfHN["thread_month"],
    "is_duplicate": dfHN["is_duplicate"],
    "is_job_post": dfHN["is_job_post"],
    "via": "Hacker News",
    "description": dfHN["description"],
})

dfHNClean.to_csv("hn_clean.csv", index=False)

print(dfHNClean.shape)
dfHNClean.isnull().sum()

(7626, 23)


,0
source,0
job_id,0
title,0
company_name,0
location,7626
state,0
region,0
is_remote,0
job_type,0
is_senior,0
